# 1-3 실습

(작성 예정)

**목차**

- 1. 실습 방법 (실습)
- 2. 01_lcel_runnable (`01_lcel_runnable.py`) (코드) (실습)
  - 2.1 코드 구조 보기 (코드)
  - 2.2 TODO: system, human 메시지 튜플을 작성해 주세요. (실습)
  - 2.3 TODO: 위에 정의한 프롬프트, 모델, 출력 파서를 파이프라인 연산자로 연결해 주세요. (실습)
  - 2.4 실행하고 솔루션과 비교하기 (코드)
- 3. 02_tool_calling (`02_tool_calling.py`) (코드) (실습)
  - 3.1 코드 구조 보기 (코드)
  - 3.2 TODO: LLM에 연결할 도구 목록을 작성해 주세요. (실습)
  - 3.3 TODO: bind_tools()를 사용해 LLM에 도구 스키마를 연결해 주세요. (실습)
  - 3.4 TODO: 도구 이름으로 실제 도구를 찾을 수 있도록 매핑을 만들어 주세요. (실습)
  - 3.5 TODO: tool_calls의 이름과 인자를 사용해 해당 도구를 실행해 주세요. (실습)
  - 3.6 실행하고 솔루션과 비교하기 (코드)
- 4. 03_output_parser (`03_output_parser.py`) (코드) (실습)
  - 4.1 코드 구조 보기 (코드)
  - 4.2 TODO: MovieReview 스키마를 사용하는 PydanticOutputParser를 생성해 주세요. (실습)
  - 4.3 TODO: format_instructions가 포함된 프롬프트를 구성해 주세요. (실습)
  - 4.4 TODO: 프롬프트, LLM, 파서를 LCEL 체인으로 연결해 주세요. (실습)
  - 4.5 TODO: with_structured_output()으로 구조화 출력을 지원하는 LLM을 만들어 주세요. (실습)
  - 4.6 TODO: 프롬프트와 구조화 LLM을 연결해 주세요. (실습)
  - 4.7 실행하고 솔루션과 비교하기 (코드)
- 5. 04_memory_runnable (`04_memory_runnable.py`) (코드) (실습)
  - 5.1 코드 구조 보기 (코드)
  - 5.2 TODO: session_id별 InMemoryChatMessageHistory를 생성하고 반환해 주세요. (실습)
  - 5.3 TODO: RunnableWithMessageHistory로 체인과 세션 히스토리를 연결해 주세요. (실습)
  - 5.4 실행하고 솔루션과 비교하기 (코드)
- 6. 05_memory_langgraph (`05_memory_langgraph.py`) (코드) (실습)
  - 6.1 코드 구조 보기 (코드)
  - 6.2 TODO: system 메시지와 State의 messages를 함께 LLM에 전달하고 결과를 반환해 주세요. (실습)
  - 6.3 TODO: chatbot 노드와 START/END 엣지를 추가해 주세요. (실습)
  - 6.4 TODO: MemorySaver를 생성하고 그래프를 컴파일해 주세요. (실습)
  - 6.5 실행하고 솔루션과 비교하기 (코드)
- 7. 정리

In [ ]:
import sys

sys.path.insert(0, '..')  # lessons/assets 도구를 불러오기 위한 경로

from assets.widgets import make_blank, run_step, show_checklist, show_code_map, show_diff

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 프로젝트의 .env 파일을 읽습니다

# 값은 출력하지 않습니다. 빠진 설정이 있는지만 봅니다.
provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
필요한_이름 = {
    "openai": ["OPENAI_API_KEY", "OPENAI_MODEL"],
    "samsung": ["api_base_url", "model", "credential_key", "send_system_name", "user_id"],
}.get(provider, ["LLM_PROVIDER"])
빠진_이름 = [name for name in 필요한_이름 if not os.getenv(name)]

import config.llm  # 실습 코드가 쓰는 설정 모듈을 불러옵니다

if 빠진_이름:
    print(f"준비가 덜 되었습니다. 빠진 설정 {len(빠진_이름)}개가 있습니다. 강사에게 알려 주세요.")
    print("(강사 확인용:", ", ".join(빠진_이름), ")")
else:
    print("준비 완료: LLM에 연결할 설정이 모두 있습니다.")

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 강사에게 알립니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

## 1. 실습 방법 <small>(실습)</small>

CodeMate에 주는 지시문은 네 칸으로 씁니다. 강의의 "좋은 입력의 네 가지"를 그대로 쓴 것입니다.

```text
[역할·목표] 무엇을 하는 사람으로서, 무엇을 만들어 달라는지
[배경·제약] 어느 파일의 어느 빈칸인지, 쓸 수 있는 것과 없는 것
[출력 형식] 코드만? 코드 + 설명 3줄?
[모호함 줄이기] 헷갈릴 만한 부분을 미리 정해 주기
[하지 말 것] 빈칸 밖은 고치지 않기, 새 기능 추가하지 않기
```

순서는 늘 같습니다.
1. 빈칸 위의 TODO 주석을 읽고, 무엇을 채워야 하는지 한 문장으로 말해 봅니다.
2. 지시문을 CodeMate 채팅에 붙여 넣습니다. 실습 파일을 열어 둔 상태에서 합니다.
3. CodeMate가 채운 코드를 읽고, 모르는 줄이 있으면 "이 줄이 하는 일"을 다시 묻습니다.
4. 실행해 봅니다. 오류가 나면 오류 메시지를 그대로 CodeMate에 붙여 넣고 원인부터 설명해 달라고 합니다.
5. 솔루션과 비교합니다. 다르다고 틀린 것은 아닙니다. 왜 다른지 말할 수 있으면 됩니다.

## 2. 01_lcel_runnable (`01_lcel_runnable.py`) <small>(코드) (실습)</small>

(작성 예정)

아래 셀을 실행하면 실습 파일이 `lessons/work/` 아래에 만들어집니다. TODO 아래 정답 줄은 빈칸(`...`)으로 바뀌어 있습니다. VS Code에서 그 파일을 열어 두고 CodeMate와 함께 채웁니다.

In [ ]:
make_blank('../../src/raw/Chapter01/02-langchain/01_lcel_runnable.py', '../work/Chapter01/02-langchain/01_lcel_runnable.py', [{'start': "('system', '너는 초보자를 위한 AI 튜터야'),", 'end': '# 사용자마다 서로 다른 내용의 주제를 질문하게 될 것이다.'},
 {'start': 'chain = prompt | model | parser # runnable', 'end': '# Runnable 실행 방식:'}])

### 2.1 코드 구조 보기 <small>(코드)</small>

In [ ]:
show_code_map('../work/Chapter01/02-langchain/01_lcel_runnable.py', [{'title': '불러오기와 준비', 'marker': '"""', 'note': '필요한 도구를 불러오고 준비합니다.'},
 {'title': '기본 LCEL 체인',
  'marker': '# ── 기본 LCEL 체인 ────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': 'RunnableParallel',
  'marker': '# ── RunnableParallel ──────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': 'RunnablePassthrough',
  'marker': '# ── RunnablePassthrough ───────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': 'RunnableLambda',
  'marker': '# ── RunnableLambda ────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'}])

### 2.2 TODO: system, human 메시지 튜플을 작성해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 2.3 TODO: 위에 정의한 프롬프트, 모델, 출력 파서를 파이프라인 연산자로 연결해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 2.4 실행하고 솔루션과 비교하기 <small>(코드)</small>

빈칸을 모두 채웠으면 실행합니다. 결과를 본 뒤 솔루션과 나란히 비교합니다.

In [ ]:
run_step('../work/Chapter01/02-langchain/01_lcel_runnable.py')

(작성 예정)

In [ ]:
show_diff('../work/Chapter01/02-langchain/01_lcel_runnable.py', '../../src/solution/Chapter01/02-langchain/01_lcel_runnable.py')

In [ ]:
show_checklist(['빈칸만 채웠고 다른 줄은 바꾸지 않았다',
 '실행했을 때 오류 없이 결과가 나왔다',
 '채운 코드의 각 줄이 하는 일을 말할 수 있다',
 '솔루션과 다른 곳이 있다면 왜 다른지 설명할 수 있다'], title='01_lcel_runnable.py 리뷰')

## 3. 02_tool_calling (`02_tool_calling.py`) <small>(코드) (실습)</small>

(작성 예정)

In [ ]:
make_blank('../../src/raw/Chapter01/02-langchain/02_tool_calling.py', '../work/Chapter01/02-langchain/02_tool_calling.py', [{'start': 'tools = [get_weather, calculate]',
  'end': '# ── LLM에 도구 바인딩 ─────────────────────────────────────────────────────────'},
 {'start': 'llm_with_tools = llm.bind_tools(tools)', 'end': '# 도구 이름 → 실행 함수 매핑'},
 {'start': "'''",
  'end': '# ── 수동 도구 호출 루프 ───────────────────────────────────────────────────────'},
 {'start': "tool_fn = tool_map[call['name']]",
  'end': '# ── 실행 예시 ─────────────────────────────────────────────────────────────────'}])

### 3.1 코드 구조 보기 <small>(코드)</small>

In [ ]:
show_code_map('../work/Chapter01/02-langchain/02_tool_calling.py', [{'title': '불러오기와 준비', 'marker': '"""', 'note': '필요한 도구를 불러오고 준비합니다.'},
 {'title': '도구 정의 (@tool 데코레이터)',
  'marker': '# ── 도구 정의 (@tool 데코레이터) ────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': 'LLM에 도구 바인딩',
  'marker': '# ── LLM에 도구 바인딩 ─────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '수동 도구 호출 루프',
  'marker': '# ── 수동 도구 호출 루프 ───────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '실행 예시',
  'marker': '# ── 실행 예시 ─────────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'}])

### 3.2 TODO: LLM에 연결할 도구 목록을 작성해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 3.3 TODO: bind_tools()를 사용해 LLM에 도구 스키마를 연결해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 3.4 TODO: 도구 이름으로 실제 도구를 찾을 수 있도록 매핑을 만들어 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 3.5 TODO: tool_calls의 이름과 인자를 사용해 해당 도구를 실행해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 3.6 실행하고 솔루션과 비교하기 <small>(코드)</small>

In [ ]:
run_step('../work/Chapter01/02-langchain/02_tool_calling.py')

(작성 예정)

In [ ]:
show_diff('../work/Chapter01/02-langchain/02_tool_calling.py', '../../src/solution/Chapter01/02-langchain/02_tool_calling.py')

In [ ]:
show_checklist(['빈칸만 채웠고 다른 줄은 바꾸지 않았다',
 '실행했을 때 오류 없이 결과가 나왔다',
 '채운 코드의 각 줄이 하는 일을 말할 수 있다',
 '솔루션과 다른 곳이 있다면 왜 다른지 설명할 수 있다'], title='02_tool_calling.py 리뷰')

## 4. 03_output_parser (`03_output_parser.py`) <small>(코드) (실습)</small>

(작성 예정)

In [ ]:
make_blank('../../src/raw/Chapter01/02-langchain/03_output_parser.py', '../work/Chapter01/02-langchain/03_output_parser.py', [{'start': 'parser = PydanticOutputParser(pydantic_object=MovieReview)',
  'end': '# TODO: format_instructions가 포함된 프롬프트를 구성해 주세요.'},
 {'start': 'prompt = ChatPromptTemplate.from_messages([',
  'end': '# TODO: 프롬프트, LLM, 파서를 LCEL 체인으로 연결해 주세요.'},
 {'start': 'chain1 = prompt | llm | parser', 'end': '# review1은 MovieReview의 인스턴스가 되길 희망한다.'},
 {'start': '', 'end': '# prompt2 = ChatPromptTemplate.from_messages(['},
 {'start': '', 'end': "# print(f'제목   : {review2.title}')"}])

### 4.1 코드 구조 보기 <small>(코드)</small>

In [ ]:
show_code_map('../work/Chapter01/02-langchain/03_output_parser.py', [{'title': '불러오기와 준비', 'marker': '"""', 'note': '필요한 도구를 불러오고 준비합니다.'},
 {'title': '출력 스키마 정의',
  'marker': '# ── 출력 스키마 정의 ───────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '방법 1: PydanticOutputParser',
  'marker': '# ── 방법 1: PydanticOutputParser ──────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '방법 2: .with_structured_output()',
  'marker': '# ── 방법 2: .with_structured_output() ────────────────────────────────────────',
  'note': '(설명 작성 예정)'}])

### 4.2 TODO: MovieReview 스키마를 사용하는 PydanticOutputParser를 생성해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 4.3 TODO: format_instructions가 포함된 프롬프트를 구성해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 4.4 TODO: 프롬프트, LLM, 파서를 LCEL 체인으로 연결해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 4.5 TODO: with_structured_output()으로 구조화 출력을 지원하는 LLM을 만들어 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 4.6 TODO: 프롬프트와 구조화 LLM을 연결해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 4.7 실행하고 솔루션과 비교하기 <small>(코드)</small>

In [ ]:
run_step('../work/Chapter01/02-langchain/03_output_parser.py')

(작성 예정)

In [ ]:
show_diff('../work/Chapter01/02-langchain/03_output_parser.py', '../../src/solution/Chapter01/02-langchain/03_output_parser.py')

In [ ]:
show_checklist(['빈칸만 채웠고 다른 줄은 바꾸지 않았다',
 '실행했을 때 오류 없이 결과가 나왔다',
 '채운 코드의 각 줄이 하는 일을 말할 수 있다',
 '솔루션과 다른 곳이 있다면 왜 다른지 설명할 수 있다'], title='03_output_parser.py 리뷰')

## 5. 04_memory_runnable (`04_memory_runnable.py`) <small>(코드) (실습)</small>

(작성 예정)

In [ ]:
make_blank('../../src/raw/Chapter01/02-langchain/04_memory_runnable.py', '../work/Chapter01/02-langchain/04_memory_runnable.py', [{'start': 'if session_id not in session_store:', 'end': '# 이미 있으면?'},
 {'start': 'chain_with_history = RunnableWithMessageHistory(',
  'end': '# ── 세션 A ────────────────────────────────────────────────────────────────────'}])

### 5.1 코드 구조 보기 <small>(코드)</small>

In [ ]:
show_code_map('../work/Chapter01/02-langchain/04_memory_runnable.py', [{'title': '불러오기와 준비', 'marker': '"""', 'note': '필요한 도구를 불러오고 준비합니다.'},
 {'title': '세션 저장소',
  'marker': '# ── 세션 저장소 ───────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '체인 구성',
  'marker': '# ── 체인 구성 ─────────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '세션 A',
  'marker': '# ── 세션 A ────────────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '세션 B',
  'marker': '# ── 세션 B ────────────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '저장된 히스토리 확인',
  'marker': '# ── 저장된 히스토리 확인 ──────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': 'LangChain 방식의 한계',
  'marker': '# ── LangChain 방식의 한계 ─────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'}])

### 5.2 TODO: session_id별 InMemoryChatMessageHistory를 생성하고 반환해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 5.3 TODO: RunnableWithMessageHistory로 체인과 세션 히스토리를 연결해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 5.4 실행하고 솔루션과 비교하기 <small>(코드)</small>

In [ ]:
run_step('../work/Chapter01/02-langchain/04_memory_runnable.py')

(작성 예정)

In [ ]:
show_diff('../work/Chapter01/02-langchain/04_memory_runnable.py', '../../src/solution/Chapter01/02-langchain/04_memory_runnable.py')

In [ ]:
show_checklist(['빈칸만 채웠고 다른 줄은 바꾸지 않았다',
 '실행했을 때 오류 없이 결과가 나왔다',
 '채운 코드의 각 줄이 하는 일을 말할 수 있다',
 '솔루션과 다른 곳이 있다면 왜 다른지 설명할 수 있다'], title='04_memory_runnable.py 리뷰')

## 6. 05_memory_langgraph (`05_memory_langgraph.py`) <small>(코드) (실습)</small>

(작성 예정)

In [ ]:
make_blank('../../src/raw/Chapter01/02-langchain/05_memory_langgraph.py', '../work/Chapter01/02-langchain/05_memory_langgraph.py', [{'start': "return {'messages': []}",
  'end': '# ── 그래프 빌드 ───────────────────────────────────────────────────────────────'},
 {'start': '', 'end': '# MemorySaver: 스레드별 State를 메모리에 유지 (프로세스 재시작 시 초기화)'},
 {'start': 'checkpointer = None',
  'end': '# ── 스레드 A: 홍길동 ───────────────────────────────────────────────────────────'}])

### 6.1 코드 구조 보기 <small>(코드)</small>

In [ ]:
show_code_map('../work/Chapter01/02-langchain/05_memory_langgraph.py', [{'title': '불러오기와 준비', 'marker': '"""', 'note': '필요한 도구를 불러오고 준비합니다.'},
 {'title': 'State 정의',
  'marker': '# ── State 정의 ────────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '그래프 빌드',
  'marker': '# ── 그래프 빌드 ───────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '스레드 A: 홍길동',
  'marker': '# ── 스레드 A: 홍길동 ───────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '스레드 B: 별개 세션',
  'marker': '# ── 스레드 B: 별개 세션 ───────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': '스냅샷 확인',
  'marker': '# ── 스냅샷 확인 ───────────────────────────────────────────────────────────────',
  'note': '(설명 작성 예정)'},
 {'title': 'LangGraph vs LangChain 방식 비교',
  'marker': '# ── LangGraph vs LangChain 방식 비교 ─────────────────────────────────────────',
  'note': '(설명 작성 예정)'}])

### 6.2 TODO: system 메시지와 State의 messages를 함께 LLM에 전달하고 결과를 반환해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 6.3 TODO: chatbot 노드와 START/END 엣지를 추가해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 6.4 TODO: MemorySaver를 생성하고 그래프를 컴파일해 주세요. <small>(실습)</small>

(작성 예정)

**CodeMate 지시문**

```text
(작성 예정)
```

(작성 예정)

### 6.5 실행하고 솔루션과 비교하기 <small>(코드)</small>

In [ ]:
run_step('../work/Chapter01/02-langchain/05_memory_langgraph.py')

(작성 예정)

In [ ]:
show_diff('../work/Chapter01/02-langchain/05_memory_langgraph.py', '../../src/solution/Chapter01/02-langchain/05_memory_langgraph.py')

In [ ]:
show_checklist(['빈칸만 채웠고 다른 줄은 바꾸지 않았다',
 '실행했을 때 오류 없이 결과가 나왔다',
 '채운 코드의 각 줄이 하는 일을 말할 수 있다',
 '솔루션과 다른 곳이 있다면 왜 다른지 설명할 수 있다'], title='05_memory_langgraph.py 리뷰')

## 7. 정리

(작성 예정)